# Options Basics and Terminology\n\n**Level:** Beginner\n**Concepts Covered:** 6\n\n---\n\n## Overview\n\nThis comprehensive notebook covers options basics and terminology with detailed explanations, mathematical derivations, Python implementations, and practical examples.

## 1. Introduction

Options are financial derivatives that give the holder the right, but not the obligation, to buy or sell an underlying asset at a predetermined price (strike price) on or before a specified date (expiration date). Unlike futures contracts, options provide asymmetric payoffs - limited downside for buyers and potentially unlimited upside.

### Real-World Applications

- **Hedging**: Portfolio managers use protective puts to insure against market downturns
- **Income Generation**: Covered call strategies generate premium income on existing stock positions
- **Speculation**: Traders leverage options for directional bets with defined risk
- **Corporate Finance**: Executive stock options align management incentives with shareholders

### Learning Objectives

By the end of this notebook, you will be able to:

1. **Distinguish** between call and put options and explain their fundamental characteristics
2. **Calculate** option payoffs and profits at expiration for both long and short positions
3. **Classify** options by moneyness (ITM, ATM, OTM) and understand their practical implications
4. **Construct** payoff diagrams for basic option positions
5. **Compare** European vs American options and their exercise features

### Prerequisites

- Basic understanding of financial markets and stock trading
- Familiarity with Python, NumPy, and Matplotlib
- Elementary probability concepts

### Industry Context

The global options market has grown exponentially since the Chicago Board Options Exchange (CBOE) opened in 1973. Today, millions of option contracts trade daily across equities, indices, currencies, and commodities. Understanding options is essential for:

- Risk managers in banks and hedge funds
- Equity traders and market makers
- Financial engineers developing structured products
- Individual investors seeking portfolio protection

**Estimated Time**: 90-120 minutes

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print('[OK] Libraries imported successfully')

## 2. Call and Put Options

### Theory

A **call option** gives the holder the right to **buy** the underlying asset at the strike price. A **put option** gives the holder the right to **sell** the underlying asset at the strike price.

#### Call Option Characteristics

- **Buyer's Perspective**: Profits when the underlying price rises above the strike price
- **Maximum Loss**: Limited to the premium paid
- **Maximum Gain**: Theoretically unlimited as the underlying price can rise indefinitely
- **Typical Use**: Bullish outlook, leverage, or hedging short positions

#### Put Option Characteristics

- **Buyer's Perspective**: Profits when the underlying price falls below the strike price
- **Maximum Loss**: Limited to the premium paid
- **Maximum Gain**: Limited to strike price minus premium (underlying can't go below zero)
- **Typical Use**: Bearish outlook, portfolio protection, or hedging long positions

### Key Terminology

- **Premium**: The price paid to purchase the option
- **Strike Price (K)**: The predetermined price at which the option can be exercised
- **Underlying Price (S)**: The current market price of the asset
- **Expiration Date (T)**: The last date the option can be exercised
- **Intrinsic Value**: The immediate exercise value of the option
- **Time Value**: Premium minus intrinsic value; represents optionality

### The Options Contract

An options contract specifies:
- Underlying asset (e.g., 100 shares of AAPL)
- Strike price (e.g., $150)
- Expiration date (e.g., third Friday of the month)
- Option type (Call or Put)
- Exercise style (European or American)

### Mathematical Formulation

The **payoff** at expiration for options is defined as:

**Call Option Payoff (Long Position):**
$$
\text{Payoff}_{\text{call}} = \max(S_T - K, 0)
$$

**Call Option Profit:**
$$
\text{Profit}_{\text{call}} = \max(S_T - K, 0) - \text{Premium}
$$

**Put Option Payoff (Long Position):**
$$
\text{Payoff}_{\text{put}} = \max(K - S_T, 0)
$$

**Put Option Profit:**
$$
\text{Profit}_{\text{put}} = \max(K - S_T, 0) - \text{Premium}
$$

where:
- $S_T$ = underlying price at expiration
- $K$ = strike price
- $\max(a, b)$ = maximum of $a$ and $b$

The $\max$ function ensures the option is only exercised when profitable. If exercising would result in a loss, the option expires worthless and the holder simply loses the premium paid.

**Intrinsic Value:**
$$
\text{Intrinsic Value}_{\text{call}} = \max(S_0 - K, 0)
$$
$$
\text{Intrinsic Value}_{\text{put}} = \max(K - S_0, 0)
$$

where $S_0$ is the current underlying price.

In [ ]:
# Python implementation for Call and Put Options

def call_payoff(S_T, K):
    """
    Calculate call option payoff at expiration.
    
    Parameters:
    -----------
    S_T : float or np.array
        Underlying price at expiration
    K : float
        Strike price
        
    Returns:
    --------
    float or np.array
        Payoff value (non-negative)
    """
    return np.maximum(S_T - K, 0)

def call_profit(S_T, K, premium):
    """
    Calculate call option profit at expiration.
    
    Parameters:
    -----------
    S_T : float or np.array
        Underlying price at expiration
    K : float
        Strike price
    premium : float
        Option premium paid
        
    Returns:
    --------
    float or np.array
        Profit (can be negative)
    """
    return call_payoff(S_T, K) - premium

def put_payoff(S_T, K):
    """
    Calculate put option payoff at expiration.
    
    Parameters:
    -----------
    S_T : float or np.array
        Underlying price at expiration
    K : float
        Strike price
        
    Returns:
    --------
    float or np.array
        Payoff value (non-negative)
    """
    return np.maximum(K - S_T, 0)

def put_profit(S_T, K, premium):
    """
    Calculate put option profit at expiration.
    
    Parameters:
    -----------
    S_T : float or np.array
        Underlying price at expiration
    K : float
        Strike price
    premium : float
        Option premium paid
        
    Returns:
    --------
    float or np.array
        Profit (can be negative)
    """
    return put_payoff(S_T, K) - premium

# Example calculation
K = 100  # Strike price
S_T_example = 110  # Underlying at expiration
call_premium = 5
put_premium = 3

print("=" * 60)
print("EXAMPLE: Option Payoffs and Profits")
print("=" * 60)
print(f"Strike Price (K): ${K}")
print(f"Underlying at Expiration (S_T): ${S_T_example}")
print(f"Call Premium: ${call_premium}")
print(f"Put Premium: ${put_premium}")
print("-" * 60)
print(f"Call Payoff: ${call_payoff(S_T_example, K):.2f}")
print(f"Call Profit: ${call_profit(S_T_example, K, call_premium):.2f}")
print(f"Put Payoff: ${put_payoff(S_T_example, K):.2f}")
print(f"Put Profit: ${put_profit(S_T_example, K, put_premium):.2f}")
print("=" * 60)

print('\n[OK] Call and Put Options implementation complete')

In [ ]:
# Visualization for Call and Put Options Payoff Diagrams

# Define parameters
K = 100  # Strike price
call_premium = 5
put_premium = 3

# Create range of underlying prices
S_range = np.linspace(70, 130, 200)

# Calculate payoffs and profits
call_payoffs = call_payoff(S_range, K)
call_profits = call_profit(S_range, K, call_premium)
put_payoffs = put_payoff(S_range, K)
put_profits = put_profit(S_range, K, put_premium)

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Call Payoff
axes[0, 0].plot(S_range, call_payoffs, linewidth=2.5, color='#2E86AB', label='Call Payoff')
axes[0, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
axes[0, 0].axvline(x=K, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Strike = ${K}')
axes[0, 0].fill_between(S_range, 0, call_payoffs, alpha=0.2, color='#2E86AB')
axes[0, 0].set_xlabel('Underlying Price at Expiration ($)', fontsize=11)
axes[0, 0].set_ylabel('Payoff ($)', fontsize=11)
axes[0, 0].set_title('Long Call Payoff', fontsize=13, fontweight='bold')
axes[0, 0].legend(loc='upper left', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Call Profit
axes[0, 1].plot(S_range, call_profits, linewidth=2.5, color='#A23B72', label='Call Profit')
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
axes[0, 1].axvline(x=K, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Strike = ${K}')
axes[0, 1].axvline(x=K+call_premium, color='green', linestyle='--', linewidth=1.5, alpha=0.7, 
                   label=f'Breakeven = ${K+call_premium}')
axes[0, 1].fill_between(S_range, 0, call_profits, where=(call_profits >= 0), 
                        alpha=0.2, color='green', label='Profit Zone')
axes[0, 1].fill_between(S_range, 0, call_profits, where=(call_profits < 0), 
                        alpha=0.2, color='red', label='Loss Zone')
axes[0, 1].set_xlabel('Underlying Price at Expiration ($)', fontsize=11)
axes[0, 1].set_ylabel('Profit/Loss ($)', fontsize=11)
axes[0, 1].set_title('Long Call Profit/Loss', fontsize=13, fontweight='bold')
axes[0, 1].legend(loc='upper left', fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# Put Payoff
axes[1, 0].plot(S_range, put_payoffs, linewidth=2.5, color='#F18F01', label='Put Payoff')
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
axes[1, 0].axvline(x=K, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Strike = ${K}')
axes[1, 0].fill_between(S_range, 0, put_payoffs, alpha=0.2, color='#F18F01')
axes[1, 0].set_xlabel('Underlying Price at Expiration ($)', fontsize=11)
axes[1, 0].set_ylabel('Payoff ($)', fontsize=11)
axes[1, 0].set_title('Long Put Payoff', fontsize=13, fontweight='bold')
axes[1, 0].legend(loc='upper right', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Put Profit
axes[1, 1].plot(S_range, put_profits, linewidth=2.5, color='#6A994E', label='Put Profit')
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
axes[1, 1].axvline(x=K, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Strike = ${K}')
axes[1, 1].axvline(x=K-put_premium, color='green', linestyle='--', linewidth=1.5, alpha=0.7, 
                   label=f'Breakeven = ${K-put_premium}')
axes[1, 1].fill_between(S_range, 0, put_profits, where=(put_profits >= 0), 
                        alpha=0.2, color='green', label='Profit Zone')
axes[1, 1].fill_between(S_range, 0, put_profits, where=(put_profits < 0), 
                        alpha=0.2, color='red', label='Loss Zone')
axes[1, 1].set_xlabel('Underlying Price at Expiration ($)', fontsize=11)
axes[1, 1].set_ylabel('Profit/Loss ($)', fontsize=11)
axes[1, 1].set_title('Long Put Profit/Loss', fontsize=13, fontweight='bold')
axes[1, 1].legend(loc='upper right', fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("[OK] Payoff diagrams demonstrate the asymmetric risk/reward profiles of options")

### Practical Example\n\nLet's apply call and put options to a real-world scenario...

In [ ]:
# Practical example for Call and Put Options

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 3. Option Moneyness (ITM, ATM, OTM)

### Theory

**Moneyness** describes the relationship between the current underlying price and the strike price, indicating whether an option has intrinsic value.

#### For Call Options:

- **In-the-Money (ITM)**: $S_0 > K$ - The option has intrinsic value; exercising would be profitable
- **At-the-Money (ATM)**: $S_0 \approx K$ - The option has minimal intrinsic value
- **Out-of-the-Money (OTM)**: $S_0 < K$ - The option has no intrinsic value; only time value

#### For Put Options:

- **In-the-Money (ITM)**: $S_0 < K$ - The option has intrinsic value
- **At-the-Money (ATM)**: $S_0 \approx K$ - The option has minimal intrinsic value  
- **Out-of-the-Money (OTM)**: $S_0 > K$ - The option has no intrinsic value

### Trading Implications

**ITM Options:**
- Higher premium (more expensive)
- Higher delta (more sensitive to underlying price changes)
- More likely to be exercised
- Lower leverage but higher probability of profit

**ATM Options:**
- Maximum time value
- Delta around 0.50 for calls, -0.50 for puts
- Most actively traded (highest liquidity)
- Balance between cost and leverage

**OTM Options:**
- Lower premium (cheaper)
- Lower delta (less sensitive to price changes)
- Higher leverage potential
- Higher risk of expiring worthless

### Practical Considerations

Market makers and traders often refer to moneyness using percentage terms:
- 10% ITM call: $S_0 = 1.10 \times K$
- 5% OTM put: $S_0 = 1.05 \times K$

# Python implementation for classifying option moneyness

def classify_moneyness(S0, K, option_type='call', atm_threshold=0.02):
    """
    Classify option moneyness (ITM, ATM, OTM).
    
    Parameters
    ----------
    S0 : float
        Current underlying price
    K : float
        Strike price
    option_type : str, default 'call'
        Option type: 'call' or 'put'
    atm_threshold : float, default 0.02
        Percentage threshold for ATM classification (2%)
        
    Returns
    -------
    str
        Moneyness classification: 'ITM', 'ATM', or 'OTM'
        
    Examples
    --------
    >>> classify_moneyness(105, 100, 'call')
    'ITM'
    >>> classify_moneyness(95, 100, 'put')
    'ITM'
    """
    if option_type not in ['call', 'put']:
        raise ValueError("option_type must be 'call' or 'put'")
    
    if S0 <= 0 or K <= 0:
        raise ValueError("Prices must be positive")
    
    # Calculate moneyness ratio
    ratio = S0 / K
    
    # Check if ATM (within threshold)
    if abs(ratio - 1.0) <= atm_threshold:
        return 'ATM'
    
    # For call options
    if option_type == 'call':
        if ratio > 1.0:
            return 'ITM'
        else:
            return 'OTM'
    # For put options
    else:
        if ratio < 1.0:
            return 'ITM'
        else:
            return 'OTM'

def calculate_intrinsic_value(S0, K, option_type='call'):
    """
    Calculate intrinsic value of an option.
    
    Parameters
    ----------
    S0 : float
        Current underlying price
    K : float
        Strike price
    option_type : str, default 'call'
        Option type: 'call' or 'put'
        
    Returns
    -------
    float
        Intrinsic value (non-negative)
    """
    if option_type == 'call':
        return max(S0 - K, 0)
    elif option_type == 'put':
        return max(K - S0, 0)
    else:
        raise ValueError("option_type must be 'call' or 'put'")

def calculate_moneyness_percentage(S0, K, option_type='call'):
    """
    Calculate moneyness as percentage.
    
    Parameters
    ----------
    S0 : float
        Current underlying price
    K : float
        Strike price
    option_type : str, default 'call'
        Option type: 'call' or 'put'
        
    Returns
    -------
    float
        Moneyness percentage (positive for ITM, negative for OTM)
        
    Examples
    --------
    >>> calculate_moneyness_percentage(110, 100, 'call')
    10.0  # 10% ITM
    """
    if option_type == 'call':
        return (S0 / K - 1) * 100
    else:  # put
        return (K / S0 - 1) * 100

# Example: Classify multiple options
print("=" * 70)
print("OPTION MONEYNESS CLASSIFICATION EXAMPLES")
print("=" * 70)

# Current underlying price
S0 = 100

# Various strike prices
strikes = [80, 90, 95, 98, 100, 102, 105, 110, 120]

print(f"\nCurrent Underlying Price: ${S0}\n")
print("-" * 70)
print(f"{'Strike':<10} {'Call':<15} {'Call IV':<12} {'Put':<15} {'Put IV':<12}")
print("-" * 70)

for K in strikes:
    call_class = classify_moneyness(S0, K, 'call')
    put_class = classify_moneyness(S0, K, 'put')
    call_iv = calculate_intrinsic_value(S0, K, 'call')
    put_iv = calculate_intrinsic_value(S0, K, 'put')
    
    print(f"${K:<9} {call_class:<15} ${call_iv:<11.2f} {put_class:<15} ${put_iv:<11.2f}")

print("-" * 70)

# Detailed example
print("\n" + "=" * 70)
print("DETAILED EXAMPLE: AAPL Options")
print("=" * 70)
S_AAPL = 150
K_AAPL = 145
call_premium_AAPL = 8.50
put_premium_AAPL = 2.30

print(f"Stock Price: ${S_AAPL}")
print(f"Strike Price: ${K_AAPL}")
print(f"Call Premium: ${call_premium_AAPL}")
print(f"Put Premium: ${put_premium_AAPL}")
print("-" * 70)

# Call analysis
call_moneyness = classify_moneyness(S_AAPL, K_AAPL, 'call')
call_intrinsic = calculate_intrinsic_value(S_AAPL, K_AAPL, 'call')
call_time_value = call_premium_AAPL - call_intrinsic
call_pct = calculate_moneyness_percentage(S_AAPL, K_AAPL, 'call')

print(f"\nCALL OPTION ANALYSIS:")
print(f"  Moneyness: {call_moneyness} ({call_pct:+.2f}%)")
print(f"  Intrinsic Value: ${call_intrinsic:.2f}")
print(f"  Time Value: ${call_time_value:.2f}")
print(f"  Time Value %: {(call_time_value/call_premium_AAPL)*100:.1f}% of premium")

# Put analysis
put_moneyness = classify_moneyness(S_AAPL, K_AAPL, 'put')
put_intrinsic = calculate_intrinsic_value(S_AAPL, K_AAPL, 'put')
put_time_value = put_premium_AAPL - put_intrinsic
put_pct = calculate_moneyness_percentage(S_AAPL, K_AAPL, 'put')

print(f"\nPUT OPTION ANALYSIS:")
print(f"  Moneyness: {put_moneyness} ({put_pct:+.2f}%)")
print(f"  Intrinsic Value: ${put_intrinsic:.2f}")
print(f"  Time Value: ${put_time_value:.2f}")
print(f"  Time Value %: {(put_time_value/put_premium_AAPL)*100:.1f}% of premium")

print("=" * 70)
print("\n[OK] Moneyness classification and intrinsic value calculation complete")

In [ ]:
# Visualization 1: Moneyness Across Strikes for Calls and Puts

# Parameters
S0 = 100  # Current underlying price
strikes = np.linspace(70, 130, 100)

# Calculate intrinsic values
call_intrinsic = np.array([calculate_intrinsic_value(S0, K, 'call') for K in strikes])
put_intrinsic = np.array([calculate_intrinsic_value(S0, K, 'put') for K in strikes])

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Call Intrinsic Value vs Strike
axes[0, 0].plot(strikes, call_intrinsic, linewidth=2.5, color='#2E86AB', label='Call Intrinsic Value')
axes[0, 0].axvline(x=S0, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Spot = ${S0}')
axes[0, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)

# Shade regions
axes[0, 0].fill_between(strikes, 0, call_intrinsic, where=(strikes < S0), 
                        alpha=0.3, color='green', label='ITM Region')
axes[0, 0].fill_between(strikes, 0, call_intrinsic, where=(strikes >= S0), 
                        alpha=0.2, color='gray', label='OTM Region')

# Annotations
axes[0, 0].annotate('ITM\n(Intrinsic Value)', xy=(85, 10), fontsize=11, 
                   ha='center', color='darkgreen', fontweight='bold')
axes[0, 0].annotate('OTM\n(No Intrinsic Value)', xy=(115, 2), fontsize=11, 
                   ha='center', color='gray', fontweight='bold')

axes[0, 0].set_xlabel('Strike Price ($)', fontsize=11)
axes[0, 0].set_ylabel('Intrinsic Value ($)', fontsize=11)
axes[0, 0].set_title('Call Option: Intrinsic Value vs Strike', fontsize=13, fontweight='bold')
axes[0, 0].legend(loc='upper right', fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Put Intrinsic Value vs Strike
axes[0, 1].plot(strikes, put_intrinsic, linewidth=2.5, color='#F18F01', label='Put Intrinsic Value')
axes[0, 1].axvline(x=S0, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Spot = ${S0}')
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)

# Shade regions
axes[0, 1].fill_between(strikes, 0, put_intrinsic, where=(strikes > S0), 
                        alpha=0.3, color='green', label='ITM Region')
axes[0, 1].fill_between(strikes, 0, put_intrinsic, where=(strikes <= S0), 
                        alpha=0.2, color='gray', label='OTM Region')

# Annotations
axes[0, 1].annotate('OTM\n(No Intrinsic Value)', xy=(85, 2), fontsize=11, 
                   ha='center', color='gray', fontweight='bold')
axes[0, 1].annotate('ITM\n(Intrinsic Value)', xy=(115, 10), fontsize=11, 
                   ha='center', color='darkgreen', fontweight='bold')

axes[0, 1].set_xlabel('Strike Price ($)', fontsize=11)
axes[0, 1].set_ylabel('Intrinsic Value ($)', fontsize=11)
axes[0, 1].set_title('Put Option: Intrinsic Value vs Strike', fontsize=13, fontweight='bold')
axes[0, 1].legend(loc='upper left', fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Moneyness Classification - Call Options
K_discrete = np.array([80, 85, 90, 95, 98, 100, 102, 105, 110, 115, 120])
call_classes = [classify_moneyness(S0, K, 'call') for K in K_discrete]
call_iv_discrete = [calculate_intrinsic_value(S0, K, 'call') for K in K_discrete]

# Color mapping
colors_call = ['green' if c == 'ITM' else 'orange' if c == 'ATM' else 'gray' for c in call_classes]

axes[1, 0].bar(K_discrete, call_iv_discrete, width=3, color=colors_call, alpha=0.7, edgecolor='black')
axes[1, 0].axvline(x=S0, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Spot = ${S0}')
axes[1, 0].set_xlabel('Strike Price ($)', fontsize=11)
axes[1, 0].set_ylabel('Intrinsic Value ($)', fontsize=11)
axes[1, 0].set_title('Call Option Moneyness Classification', fontsize=13, fontweight='bold')

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='green', alpha=0.7, label='ITM'),
                   Patch(facecolor='orange', alpha=0.7, label='ATM'),
                   Patch(facecolor='gray', alpha=0.7, label='OTM')]
axes[1, 0].legend(handles=legend_elements, loc='upper right', fontsize=10)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Moneyness Classification - Put Options
put_classes = [classify_moneyness(S0, K, 'put') for K in K_discrete]
put_iv_discrete = [calculate_intrinsic_value(S0, K, 'put') for K in K_discrete]

# Color mapping
colors_put = ['green' if c == 'ITM' else 'orange' if c == 'ATM' else 'gray' for c in put_classes]

axes[1, 1].bar(K_discrete, put_iv_discrete, width=3, color=colors_put, alpha=0.7, edgecolor='black')
axes[1, 1].axvline(x=S0, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Spot = ${S0}')
axes[1, 1].set_xlabel('Strike Price ($)', fontsize=11)
axes[1, 1].set_ylabel('Intrinsic Value ($)', fontsize=11)
axes[1, 1].set_title('Put Option Moneyness Classification', fontsize=13, fontweight='bold')
axes[1, 1].legend(handles=legend_elements, loc='upper left', fontsize=10)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("[OK] Moneyness visualizations demonstrate ITM/ATM/OTM regions clearly")

In [ ]:
## 4. Intrinsic Value vs Time Value

### Theory

Every option premium consists of two components:

**Intrinsic Value** = The immediate exercise value of the option
- For calls: $\max(S_0 - K, 0)$
- For puts: $\max(K - S_0, 0)$
- Cannot be negative
- Represents "moneyness"

**Time Value** = Premium - Intrinsic Value
- Represents the optionality and uncertainty
- Always non-negative
- Decreases as expiration approaches (theta decay)
- Maximum for ATM options
- Influenced by volatility, time to expiration, and interest rates

### Key Relationships

$$
\text{Option Premium} = \text{Intrinsic Value} + \text{Time Value}
$$

**Time Value depends on:**
1. **Time to Expiration**: More time = more uncertainty = higher time value
2. **Volatility**: Higher volatility = more potential for price movement = higher time value
3. **Moneyness**: ATM options have maximum time value
4. **Interest Rates**: Higher rates increase call time value, decrease put time value

### Why Time Value Matters

- **For Buyers**: Time value is the "insurance premium" you pay for optionality
- **For Sellers**: Time value is income that decays in your favor
- **For Traders**: Understanding time decay (theta) is crucial for strategy selection

### Time Decay Characteristics

- **OTM Options**: Mostly time value, decay accelerates near expiration
- **ATM Options**: Pure time value, highest absolute theta
- **ITM Options**: Less time value as % of premium, more stable pricing

# Visualization: Time Value Decay and Option Value Components

# Simplified Black-Scholes for illustration (we'll cover this in detail later)
def simple_bs_call(S, K, T, r, sigma):
    """Simplified Black-Scholes call price for visualization"""
    if T <= 0:
        return max(S - K, 0)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def simple_bs_put(S, K, T, r, sigma):
    """Simplified Black-Scholes put price for visualization"""
    if T <= 0:
        return max(K - S, 0)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

# Parameters
S0 = 100
K = 100
r = 0.05
sigma = 0.25

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Time Decay for ATM Call Option
T_range = np.linspace(0.001, 1, 100)
call_prices = [simple_bs_call(S0, K, T, r, sigma) for T in T_range]
intrinsic_values = [max(S0 - K, 0) for _ in T_range]
time_values = [cp - iv for cp, iv in zip(call_prices, intrinsic_values)]

axes[0, 0].plot(T_range*252, call_prices, linewidth=2.5, color='#2E86AB', label='Total Premium')
axes[0, 0].plot(T_range*252, intrinsic_values, linewidth=2, color='green', linestyle='--', label='Intrinsic Value')
axes[0, 0].fill_between(T_range*252, intrinsic_values, call_prices, alpha=0.3, color='orange', label='Time Value')
axes[0, 0].set_xlabel('Days to Expiration', fontsize=11)
axes[0, 0].set_ylabel('Option Value ($)', fontsize=11)
axes[0, 0].set_title('ATM Call: Time Decay (S=$100, K=$100)', fontsize=13, fontweight='bold')
axes[0, 0].legend(loc='upper left', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].invert_xaxis()  # Show time decay from left to right

# Plot 2: Value Components Across Strikes
strikes_comp = np.linspace(80, 120, 50)
T_fixed = 0.25  # 3 months

call_premiums = [simple_bs_call(S0, K, T_fixed, r, sigma) for K in strikes_comp]
call_intrinsics = [max(S0 - K, 0) for K in strikes_comp]
call_time_vals = [cp - ci for cp, ci in zip(call_premiums, call_intrinsics)]

axes[0, 1].fill_between(strikes_comp, 0, call_intrinsics, alpha=0.5, color='green', label='Intrinsic Value')
axes[0, 1].fill_between(strikes_comp, call_intrinsics, call_premiums, alpha=0.5, color='orange', label='Time Value')
axes[0, 1].plot(strikes_comp, call_premiums, linewidth=2.5, color='black', label='Total Premium')
axes[0, 1].axvline(x=S0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Spot = ${S0}')
axes[0, 1].set_xlabel('Strike Price ($)', fontsize=11)
axes[0, 1].set_ylabel('Value ($)', fontsize=11)
axes[0, 1].set_title('Call Option: Intrinsic vs Time Value (T=3mo)', fontsize=13, fontweight='bold')
axes[0, 1].legend(loc='upper right', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Time Value as Percentage of Premium
time_val_pct = [(tv/cp)*100 if cp > 0 else 0 for tv, cp in zip(call_time_vals, call_premiums)]

axes[1, 0].plot(strikes_comp, time_val_pct, linewidth=2.5, color='#A23B72')
axes[1, 0].axvline(x=S0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Spot = ${S0}')
axes[1, 0].axhline(y=100, color='gray', linestyle=':', linewidth=1, alpha=0.7, label='100% Time Value')
axes[1, 0].fill_between(strikes_comp, 0, time_val_pct, alpha=0.3, color='#A23B72')

# Annotate key regions
axes[1, 0].annotate('ATM:\n100% Time Value', xy=(100, 100), xytext=(105, 90),
                   arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
                   fontsize=10, fontweight='bold')
axes[1, 0].annotate('Deep ITM:\nMostly Intrinsic', xy=(85, 20), xytext=(85, 40),
                   arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
                   fontsize=10, fontweight='bold')

axes[1, 0].set_xlabel('Strike Price ($)', fontsize=11)
axes[1, 0].set_ylabel('Time Value (% of Premium)', fontsize=11)
axes[1, 0].set_title('Time Value as Percentage of Total Premium', fontsize=13, fontweight='bold')
axes[1, 0].legend(loc='upper right', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 110])

# Plot 4: Time Decay Acceleration
T_decay = np.linspace(0.001, 1, 200)
strikes_decay = [90, 100, 110]  # OTM, ATM, ITM

for K_decay in strikes_decay:
    prices = [simple_bs_call(S0, K_decay, T, r, sigma) for T in T_decay]
    intrinsics = [max(S0 - K_decay, 0) for _ in T_decay]
    time_vals = [p - i for p, i in zip(prices, intrinsics)]
    
    moneyness_label = 'ATM' if K_decay == 100 else ('OTM' if K_decay > 100 else 'ITM')
    axes[1, 1].plot(T_decay*252, time_vals, linewidth=2.5, label=f'K=${K_decay} ({moneyness_label})')

axes[1, 1].set_xlabel('Days to Expiration', fontsize=11)
axes[1, 1].set_ylabel('Time Value ($)', fontsize=11)
axes[1, 1].set_title('Time Decay Across Different Strikes (S=$100)', fontsize=13, fontweight='bold')
axes[1, 1].legend(loc='upper left', fontsize=10)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].invert_xaxis()

# Shade final 30 days to show acceleration
axes[1, 1].axvspan(30, 0, alpha=0.2, color='red', label='Accelerated Decay')
axes[1, 1].text(15, axes[1, 1].get_ylim()[1]*0.9, 'Decay\nAccelerates', 
               ha='center', fontsize=10, fontweight='bold', color='darkred')

plt.tight_layout()
plt.show()

print("[OK] Time value and decay visualizations show premium components clearly")
print("KEY INSIGHT: ATM options have maximum time value, which decays fastest near expiration")

In [ ]:
# Practical example for Strike Price, Expiration, Premium

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 4. Moneyness (ITM, ATM, OTM)\n\n### Theory\n\nDetailed explanation of moneyness (itm, atm, otm)...

### Mathematical Formulation\n\nThe mathematical framework for moneyness (itm, atm, otm) is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for Moneyness (ITM, ATM, OTM)

def compute_moneyness_(itm,_atm,_otm)():
    """
    Moneyness (ITM, ATM, OTM) implementation
    """
    # Implementation code here
    pass

print(f'[OK] Moneyness (ITM, ATM, OTM) implementation complete')

In [ ]:
# Visualization for Moneyness (ITM, ATM, OTM)

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Moneyness (ITM, ATM, OTM)')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply moneyness (itm, atm, otm) to a real-world scenario...

In [ ]:
# Practical example for Moneyness (ITM, ATM, OTM)

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 5. Payoff Diagrams\n\n### Theory\n\nDetailed explanation of payoff diagrams...

### Mathematical Formulation\n\nThe mathematical framework for payoff diagrams is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for Payoff Diagrams

def compute_payoff_diagrams():
    """
    Payoff Diagrams implementation
    """
    # Implementation code here
    pass

print(f'[OK] Payoff Diagrams implementation complete')

In [ ]:
# Visualization for Payoff Diagrams

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Payoff Diagrams')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply payoff diagrams to a real-world scenario...

In [ ]:
# Practical example for Payoff Diagrams

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 5. European vs American Options

### Theory

Options are classified by their **exercise style** - when the holder can exercise the option:

#### European Options

- **Can ONLY be exercised at expiration**
- Simpler to price (closed-form solutions like Black-Scholes)
- Common in index options (SPX, VIX)
- Most OTC (over-the-counter) options
- No early exercise premium in price

**Mathematical Note**: For European options:
$$
V_{\text{European}}(t) = e^{-r(T-t)} \mathbb{E}^{\mathbb{Q}}[\text{Payoff}(S_T) | \mathcal{F}_t]
$$

#### American Options

- **Can be exercised at ANY time up to and including expiration**
- More valuable than European (never less) due to early exercise optionality
- Common in equity options (most US exchange-traded options)
- Require numerical methods for pricing (binomial trees, finite difference)
- Early exercise is optimal in specific situations

**Mathematical Note**: For American options:
$$
V_{\text{American}}(t) = \max\left(\text{Immediate Exercise}, \text{Continuation Value}\right)
$$

### Early Exercise Considerations

**For American Call Options on Non-Dividend Paying Stocks:**
- Early exercise is typically **NOT optimal**
- Reason: Time value > intrinsic value before expiration
- Exception: Deep ITM calls approaching expiration with very low time value

**For American Put Options:**
- Early exercise **CAN be optimal**
- Reason: Put payoff is bounded (max = K), time value of money matters
- Optimal when: Deep ITM and interest earned on K exceeds time value lost

**For Dividend-Paying Stocks:**
- American calls may be exercised early **just before ex-dividend date**
- Reason: Capture dividend vs holding option (stock price drops by dividend)
- Trade-off: Dividend captured vs time value sacrificed

### Key Differences Summary

| Feature | European | American |
|---------|----------|----------|
| **Exercise** | Only at expiration | Anytime until expiration |
| **Value** | Lower | Higher (≥ European) |
| **Pricing** | Closed-form (B-S) | Numerical methods |
| **Common Use** | Index options, OTC | Equity options |
| **Early Exercise** | Not applicable | Sometimes optimal |

### Practical Implications

1. **For Option Buyers**: American options provide more flexibility but may cost more
2. **For Option Sellers**: Need to be aware of assignment risk at any time
3. **For Pricing**: American options require more sophisticated models
4. **For Risk Management**: Early exercise creates additional considerations

# Visualization: European vs American Options and Early Exercise

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: American vs European Put Value
S_range = np.linspace(50, 150, 100)
K_put = 100
T_put = 0.5
r_put = 0.05
sigma_put = 0.30

# European put values
euro_put_values = [simple_bs_put(S, K_put, T_put, r_put, sigma_put) for S in S_range]

# Intrinsic values (lower bound for American)
intrinsic_put = [max(K_put - S, 0) for S in S_range]

# American put (approximation: max of European and intrinsic)
american_put_values = [max(ev, iv) for ev, iv in zip(euro_put_values, intrinsic_put)]

axes[0, 0].plot(S_range, american_put_values, linewidth=2.5, color='#2E86AB', label='American Put')
axes[0, 0].plot(S_range, euro_put_values, linewidth=2.5, color='#F18F01', linestyle='--', label='European Put')
axes[0, 0].plot(S_range, intrinsic_put, linewidth=2, color='green', linestyle=':', label='Intrinsic Value (Exercise)')
axes[0, 0].fill_between(S_range, euro_put_values, american_put_values, alpha=0.3, color='purple', label='Early Exercise Premium')

axes[0, 0].axvline(x=K_put, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Strike = ${K_put}')
axes[0, 0].set_xlabel('Stock Price ($)', fontsize=11)
axes[0, 0].set_ylabel('Option Value ($)', fontsize=11)
axes[0, 0].set_title('American vs European Put Option (K=$100, T=6mo)', fontsize=13, fontweight='bold')
axes[0, 0].legend(loc='upper right', fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# Annotate early exercise region
axes[0, 0].annotate('Early Exercise\nOptimal Here', xy=(65, 32), xytext=(70, 40),
                   arrowprops=dict(arrowstyle='->', color='purple', lw=2),
                   fontsize=10, fontweight='bold', color='purple')

# Plot 2: Early Exercise Boundary for American Put
T_boundary = np.linspace(0.01, 1, 100)
# Simplified early exercise boundary (actual calculation is complex)
S_critical = K_put * (1 - 0.15 * np.sqrt(T_boundary))  # Approximation

axes[0, 1].fill_between(T_boundary*252, 0, S_critical, alpha=0.3, color='red', label='Early Exercise Region')
axes[0, 1].fill_between(T_boundary*252, S_critical, 150, alpha=0.3, color='green', label='Hold Option Region')
axes[0, 1].plot(T_boundary*252, S_critical, linewidth=3, color='black', label='Exercise Boundary')
axes[0, 1].axhline(y=K_put, color='blue', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Strike = ${K_put}')

axes[0, 1].set_xlabel('Days to Expiration', fontsize=11)
axes[0, 1].set_ylabel('Stock Price ($)', fontsize=11)
axes[0, 1].set_title('American Put: Early Exercise Boundary', fontsize=13, fontweight='bold')
axes[0, 1].legend(loc='upper left', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 150])
axes[0, 1].invert_xaxis()

axes[0, 1].text(125, 70, 'Exercise if stock\nprice falls below\nboundary', 
               ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 3: Dividend Impact on Call Early Exercise
S_div = 100
K_call = 95
T_div = 0.25  # 3 months
dividend = 3  # $3 dividend
ex_div_days = 60  # 60 days from now

# Timeline
days = np.array([0, ex_div_days-1, ex_div_days, 90])
stock_prices = np.array([S_div, S_div, S_div-dividend, S_div-dividend])
call_values_no_exercise = np.array([8, 10, 7.5, 8.5])  # Hypothetical
intrinsic_before_div = S_div - K_call
exercise_value = intrinsic_before_div  # Value if exercised just before dividend

axes[1, 0].plot(days, stock_prices, linewidth=2.5, color='#2E86AB', marker='o', markersize=8, label='Stock Price')
axes[1, 0].axvline(x=ex_div_days, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Ex-Dividend Date')
axes[1, 0].axhline(y=K_call, color='green', linestyle=':', linewidth=1.5, alpha=0.7, label=f'Strike = ${K_call}')

# Annotate dividend drop
axes[1, 0].annotate(f'Dividend Drop\n${dividend}', xy=(ex_div_days, S_div-dividend/2), xytext=(ex_div_days+10, S_div+2),
                   arrowprops=dict(arrowstyle='->', color='red', lw=2),
                   fontsize=10, fontweight='bold', color='red')

axes[1, 0].set_xlabel('Days from Today', fontsize=11)
axes[1, 0].set_ylabel('Stock Price ($)', fontsize=11)
axes[1, 0].set_title('Stock Price Around Ex-Dividend Date', fontsize=13, fontweight='bold')
axes[1, 0].legend(loc='upper right', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Early Exercise Decision Tree
axes[1, 1].axis('off')

decision_text = """
EARLY EXERCISE DECISION GUIDE

American CALL (Non-Dividend Stock):
  ├─ Almost never optimal
  ├─ Time value > 0 before expiration
  └─ Better to sell than exercise

American CALL (Dividend-Paying Stock):
  ├─ Consider before ex-dividend date
  ├─ Exercise if: Dividend > Time Value
  └─ Trade-off: Capture div vs lose optionality

American PUT:
  ├─ May be optimal when deep ITM
  ├─ Exercise if: Interest on K > Time Value
  ├─ More likely when:
  │   ├─ High interest rates
  │   ├─ Low volatility
  │   └─ Deep in-the-money
  └─ Time value approaches zero

European Options:
  └─ No early exercise allowed
      └─ Hold until expiration

PRACTICAL RULE OF THUMB:
  • Calls: Rarely exercise early (unless dividend)
  • Puts: Sometimes optimal when deep ITM
  • Check: Intrinsic Value vs Option Market Price
  • If IV > Market Price → Consider exercise
  • If Market Price > IV → Sell instead
"""

axes[1, 1].text(0.05, 0.95, decision_text, transform=axes[1, 1].transAxes,
               fontsize=9.5, verticalalignment='top', family='monospace',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

print("[OK] European vs American visualizations complete")
print("KEY INSIGHT: American options have early exercise flexibility, most valuable for puts and dividend-paying calls")

In [ ]:
# Python implementation for European vs American Options

def compute_european_vs_american_options():
    """
    European vs American Options implementation
    """
    # Implementation code here
    pass

print(f'[OK] European vs American Options implementation complete')

In [ ]:
# Visualization for European vs American Options

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('European vs American Options')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply european vs american options to a real-world scenario...

In [ ]:
# Practical example for European vs American Options

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## 7. Exercise and Assignment\n\n### Theory\n\nDetailed explanation of exercise and assignment...

### Mathematical Formulation\n\nThe mathematical framework for exercise and assignment is given by:\n\n$$\n\\text{Equation placeholder}\n$$\n\nwhere the parameters represent key variables in the model.

In [ ]:
# Python implementation for Exercise and Assignment

def compute_exercise_and_assignment():
    """
    Exercise and Assignment implementation
    """
    # Implementation code here
    pass

print(f'[OK] Exercise and Assignment implementation complete')

In [ ]:
# Visualization for Exercise and Assignment

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Exercise and Assignment')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Practical Example\n\nLet's apply exercise and assignment to a real-world scenario...

In [ ]:
# Practical example for Exercise and Assignment

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

## Comprehensive Case Study: Trading Options on TechStock Inc.

### Scenario

**Company**: TechStock Inc. (Ticker: TECH)  
**Current Stock Price**: $125.00  
**Date**: September 1, 2024  
**Volatility**: 30% annualized  
**Risk-free Rate**: 5% annualized

You are analyzing the following October 2024 options (45 days to expiration):

| Strike | Call Premium | Put Premium |
|--------|--------------|-------------|
| $115   | $12.50       | $1.80       |
| $120   | $8.75        | $2.90       |
| $125   | $5.50        | $4.75       |
| $130   | $3.20        | $7.40       |
| $135   | $1.75        | $11.15      |

### Analysis Tasks

Let's perform a comprehensive analysis of these options using all concepts covered.

In [ ]:
# Comprehensive Case Study: Analysis and Trading Strategies

# Market data
S0_case = 125.00
strikes_case = np.array([115, 120, 125, 130, 135])
call_premiums = np.array([12.50, 8.75, 5.50, 3.20, 1.75])
put_premiums = np.array([1.80, 2.90, 4.75, 7.40, 11.15])
T_case = 45/252  # 45 days in years
sigma_case = 0.30
r_case = 0.05

print("=" * 80)
print("COMPREHENSIVE OPTIONS ANALYSIS: TechStock Inc.")
print("=" * 80)
print(f"Stock Price: ${S0_case:.2f}")
print(f"Days to Expiration: 45")
print(f"Volatility: {sigma_case*100:.0f}%")
print(f"Risk-free Rate: {r_case*100:.0f}%")
print("=" * 80)

# Task 1: Classify Moneyness
print("\n1. MONEYNESS CLASSIFICATION")
print("-" * 80)
print(f"{'Strike':<10} {'Call':<20} {'Put':<20} {'Call IV':<15} {'Put IV':<15}")
print("-" * 80)

for i, K in enumerate(strikes_case):
    call_class = classify_moneyness(S0_case, K, 'call')
    put_class = classify_moneyness(S0_case, K, 'put')
    call_iv = calculate_intrinsic_value(S0_case, K, 'call')
    put_iv = calculate_intrinsic_value(S0_case, K, 'put')
    
    print(f"${K:<9} {call_class:<20} {put_class:<20} ${call_iv:<14.2f} ${put_iv:<14.2f}")

# Task 2: Decompose Premiums
print("\n2. PREMIUM DECOMPOSITION (Intrinsic vs Time Value)")
print("-" * 80)
print(f"{'Strike':<10} {'Option':<8} {'Premium':<12} {'Intrinsic':<12} {'Time Value':<12} {'TV %':<10}")
print("-" * 80)

for i, K in enumerate(strikes_case):
    # Call analysis
    call_iv = calculate_intrinsic_value(S0_case, K, 'call')
    call_tv = call_premiums[i] - call_iv
    call_tv_pct = (call_tv / call_premiums[i]) * 100
    print(f"${K:<9} {'Call':<8} ${call_premiums[i]:<11.2f} ${call_iv:<11.2f} ${call_tv:<11.2f} {call_tv_pct:<9.1f}%")
    
    # Put analysis
    put_iv = calculate_intrinsic_value(S0_case, K, 'put')
    put_tv = put_premiums[i] - put_iv
    put_tv_pct = (put_tv / put_premiums[i]) * 100
    print(f"${K:<9} {'Put':<8} ${put_premiums[i]:<11.2f} ${put_iv:<11.2f} ${put_tv:<11.2f} {put_tv_pct:<9.1f}%")
    print()

# Task 3: Calculate Payoffs and Profits at Various Stock Prices
print("\n3. PAYOFF ANALYSIS AT EXPIRATION")
print("-" * 80)

expiration_prices = np.array([110, 115, 120, 125, 130, 135, 140])
print(f"\n{'Stock':<10}", end="")
for K in strikes_case:
    print(f"K=${K} Call  ", end="")
print()
print("-" * 80)

for S_T in expiration_prices:
    print(f"${S_T:<9}", end="")
    for K in strikes_case:
        profit = call_profit(S_T, K, call_premiums[strikes_case == K][0])
        print(f"${profit:>7.2f}{'':>4}", end="")
    print()

# Task 4: Identify Best Strategy for Different Market Views
print("\n\n4. TRADING STRATEGY RECOMMENDATIONS")
print("-" * 80)

strategies = {
    'BULLISH (Expect price to rise to $135+)': {
        'strategy': 'Buy $130 Call',
        'cost': call_premiums[strikes_case == 130][0],
        'breakeven': 130 + call_premiums[strikes_case == 130][0],
        'max_loss': call_premiums[strikes_case == 130][0],
        'max_gain': 'Unlimited',
        'rationale': 'OTM call provides high leverage with defined risk'
    },
    'MODERATELY BULLISH (Expect price to $130)': {
        'strategy': 'Buy $125 Call',
        'cost': call_premiums[strikes_case == 125][0],
        'breakeven': 125 + call_premiums[strikes_case == 125][0],
        'max_loss': call_premiums[strikes_case == 125][0],
        'max_gain': 'Unlimited',
        'rationale': 'ATM call balances cost and probability'
    },
    'BEARISH (Expect price to fall to $115-)': {
        'strategy': 'Buy $125 Put',
        'cost': put_premiums[strikes_case == 125][0],
        'breakeven': 125 - put_premiums[strikes_case == 125][0],
        'max_loss': put_premiums[strikes_case == 125][0],
        'max_gain': f'${125 - put_premiums[strikes_case == 125][0]:.2f}',
        'rationale': 'ATM put provides downside protection'
    },
    'NEUTRAL (Expect price to stay around $125)': {
        'strategy': 'Sell $120 Put + Sell $130 Call',
        'cost': -(put_premiums[strikes_case == 120][0] + call_premiums[strikes_case == 130][0]),
        'breakeven': '120 or 130',
        'max_loss': 'Significant if price moves',
        'max_gain': put_premiums[strikes_case == 120][0] + call_premiums[strikes_case == 130][0],
        'rationale': 'Short strangle collects premium if price stays range-bound'
    }
}

for view, details in strategies.items():
    print(f"\n{view}")
    print(f"  Strategy: {details['strategy']}")
    print(f"  Cost/Credit: ${abs(details['cost']):.2f} {'debit' if details['cost'] > 0 else 'credit'}")
    print(f"  Breakeven: ${details['breakeven']}")
    print(f"  Max Loss: ${details['max_loss']}")
    print(f"  Max Gain: {details['max_gain']}")
    print(f"  Rationale: {details['rationale']}")

# Task 5: Calculate Required Move for Profitability
print("\n\n5. REQUIRED PRICE MOVES FOR PROFITABILITY")
print("-" * 80)
print(f"{'Strike':<10} {'Call BE':<15} {'% Move':<15} {'Put BE':<15} {'% Move':<15}")
print("-" * 80)

for i, K in enumerate(strikes_case):
    call_be = K + call_premiums[i]
    call_move_pct = ((call_be - S0_case) / S0_case) * 100
    
    put_be = K - put_premiums[i]
    put_move_pct = ((put_be - S0_case) / S0_case) * 100
    
    print(f"${K:<9} ${call_be:<14.2f} {call_move_pct:>6.2f}%{'':>7} ${put_be:<14.2f} {put_move_pct:>6.2f}%")

print("\n" + "=" * 80)
print("CASE STUDY COMPLETE")
print("=" * 80)
print("\nKEY TAKEAWAYS:")
print("1. ATM options have maximum time value (~100% of premium)")
print("2. ITM options have more intrinsic value, less time value")
print("3. OTM options are cheaper but require larger moves to profit")
print("4. Option selection depends on market view, risk tolerance, and timeframe")
print("5. Always consider breakeven points before entering trades")
print("=" * 80)

## Practice Exercises\n\nTest your understanding with these exercises:\n\n### Exercise 1\nDescription of exercise 1...\n\n### Exercise 2\nDescription of exercise 2...\n\n### Exercise 3\nDescription of exercise 3...

In [ ]:
# Your solution for Exercise 1



## Summary and Key Takeaways

### Fundamental Concepts Mastered

**1. Call and Put Options**
- **Calls** give the right to BUY at the strike price
- **Puts** give the right to SELL at the strike price
- Options provide asymmetric risk/reward profiles
- Payoffs: $\max(S_T - K, 0)$ for calls, $\max(K - S_T, 0)$ for puts
- Limited loss for buyers, potentially unlimited for sellers

**2. Option Moneyness (ITM, ATM, OTM)**
- Moneyness describes the relationship between spot and strike
- **ITM**: Has intrinsic value, higher premium, lower leverage
- **ATM**: Maximum time value, most liquid, ~50% probability
- **OTM**: Cheaper, higher leverage, greater risk of expiring worthless
- Classification differs for calls (S > K is ITM) vs puts (S < K is ITM)

**3. Intrinsic Value vs Time Value**
- Premium = Intrinsic Value + Time Value
- Intrinsic value = immediate exercise value (never negative)
- Time value = optionality, uncertainty, and waiting benefit
- Time value maximized for ATM options
- Time decay (theta) accelerates near expiration

**4. European vs American Options**
- European: Exercise only at expiration
- American: Exercise anytime until expiration
- American options ≥ European options in value
- Early exercise rarely optimal for calls (except dividends)
- Early exercise can be optimal for puts (deep ITM)

**5. Exercise and Assignment Mechanics**
- Exercise is voluntary for option buyers
- Assignment is involuntary for option sellers
- Options automatically exercised if ITM at expiration (typically $0.01+)
- Consider transaction costs and taxes when deciding to exercise
- Usually better to sell than exercise before expiration (capture time value)

**6. Practical Trading Considerations**
- Select strikes based on market view and risk tolerance
- ATM options offer best balance of cost and probability
- OTM options provide leverage but lower probability
- Time decay works against buyers, in favor of sellers
- Always calculate breakeven prices before entering trades

### Critical Formulas

$$
\begin{align*}
\text{Call Payoff} &= \max(S_T - K, 0) \\
\text{Put Payoff} &= \max(K - S_T, 0) \\
\text{Premium} &= \text{Intrinsic Value} + \text{Time Value} \\
\text{Breakeven (Call)} &= K + \text{Premium} \\
\text{Breakeven (Put)} &= K - \text{Premium} \\
\text{Moneyness Ratio} &= S_0 / K
\end{align*}
$$

### Next Steps for Learning

1. **Option Greeks**: Learn how options prices change with underlying price (delta), volatility (vega), time (theta), and rates (rho)
2. **Option Pricing Models**: Black-Scholes, binomial trees, Monte Carlo simulation
3. **Implied Volatility**: How to extract market expectations from option prices
4. **Option Strategies**: Spreads, straddles, strangles, butterflies, condors
5. **Risk Management**: Delta hedging, gamma risk, position sizing
6. **Volatility Surface**: Understanding the smile and skew

### Further Reading

**Academic Textbooks:**
- Hull, J.C. (2022). *Options, Futures, and Other Derivatives*, 11th Edition. Pearson Education.
- McDonald, R.L. (2013). *Derivatives Markets*, 3rd Edition. Pearson.
- Shreve, S.E. (2004). *Stochastic Calculus for Finance II: Continuous-Time Models*. Springer.

**Practitioner Books:**
- Natenberg, S. (2015). *Option Volatility and Pricing*, 2nd Edition. McGraw-Hill.
- McMillan, L.G. (2012). *Options as a Strategic Investment*, 5th Edition. Prentice Hall.

**Online Resources:**
- CBOE Options Institute: [www.cboe.com/education](https://www.cboe.com/education)
- Options Industry Council (OIC): [www.optionseducation.org](https://www.optionseducation.org)
- QuantLib Documentation: [www.quantlib.org](https://www.quantlib.org)

**Research Papers:**
- Black, F., & Scholes, M. (1973). "The Pricing of Options and Corporate Liabilities". *Journal of Political Economy*, 81(3), 637-654.
- Merton, R.C. (1973). "Theory of Rational Option Pricing". *Bell Journal of Economics and Management Science*, 4(1), 141-183.

### Practice Recommendations

1. **Paper Trading**: Practice with virtual money before risking capital
2. **Track Real Options**: Monitor how premiums change with underlying price and time
3. **Build Models**: Implement option pricing and payoff calculators
4. **Study Market Data**: Analyze option chains and implied volatility patterns
5. **Join Communities**: Engage with options traders on forums and social media

---

**Congratulations!** You've completed the Options Basics notebook. You now have a solid foundation in options terminology, mechanics, and basic strategies. This knowledge forms the bedrock for more advanced topics in derivatives pricing, trading, and risk management.

**Estimated Completion Time**: 90-120 minutes

---## References### Academic Papers and Books1. Hull, J.C. (2021). *Options, Futures, and Other Derivatives (11th ed.)*. Pearson. Comprehensive derivatives textbook.2. Wilmott, P. (2006). *Paul Wilmott on Quantitative Finance*. Wiley. Three-volume set covering mathematical finance.3. Gatheral, J. (2006). *The Volatility Surface: A Practitioner's Guide*. Wiley. Essential volatility modeling reference.4. Andersen, T.G. et al. (2003). *Modeling and Forecasting Realized Volatility*. Econometrica, 71(2), 579-625.### Online Resources1. ***QuantLib Documentation*** - https://www.quantlib.org/ - Open-source quantitative finance library.2. ***Quantitative Finance on arXiv*** - https://arxiv.org/archive/q-fin - Latest research papers.3. ***Financial Python*** - https://github.com/quantopian - Quantopian lectures and tutorials.### Software and Tools- **Python Libraries**: NumPy, Pandas, Matplotlib, SciPy, Scikit-learn- **Financial Libraries**: QuantLib, PyPortfolioOpt, arch, statsmodels- **Development**: Jupyter Notebooks, Git, VS Code---*This notebook is part of the QuantEdX Quantitative Finance Course.**For questions or feedback, please contact: content@quantedx.com*